# 리뷰 텍스트 전처리 - Step 2: 중복 처리 (revised)

`reviews_step1_cleaned.csv`를 입력으로 받아 같은 작성자의 다중 리뷰 작성을 식별하고 처리한다.

## 처리 전략

| Step | 조건 | 처리 |
|------|------|------|
| 0 | 작성자 = `'-'` (탈퇴/닉변) | 각 행을 고유 ID로 분리 |
| 1 | 리뷰타입 = `month` | 중복 검토 제외, 그대로 보존 |
| 2-A | 같은 옵션 + 같은 타입 + 24h 이내 + 유사도 ≥ 0.85 | 1개 보존, `same_option_same_type_dup` |
| 2-A' | 같은 옵션 + 같은 타입 + 24h 이내 + 유사도 < 0.85 | 둘 다 보존, `same_option_same_type_unique` |
| 2-B | 다른 옵션 + 같은 타입 + 24h 이내 + 유사도 ≥ 0.85 | 1개 보존, `multi_option_dup` |
| 2-C | 다른 옵션 + 같은 타입 + 24h 이내 + 유사도 < 0.85 | 둘 다 보존, `multi_option_unique` |
| 3 | 다른 타입 + 24h 이내 + 유사도 ≥ 0.85 | 1개 보존, `multi_type_dup` |
| 4 | 다른 타입 + 24h 이내 + 유사도 < 0.85 | 둘 다 보존, `multi_type_unique` |
| 5 | 24h 초과 | `long_gap_review` 플래그 (재구매 가능성, 확정 불가) |

**변경 사항 (피드백 반영)**
- ① `same_option_same_type`도 유사도 체크 후 drop/keep 결정 + `_unique` 플래그 신규
- ② 세션 분리: 인접 간격 기준 → 세션 시작 기준 24h 윈도우 (체이닝 방지)
- ③ `is_repurchase` → `long_gap_review` (재구매 확정 불가)
- ④ base 선정 기준: `한글_글자수` → `전체_글자수` (숫자/영문 포함)
- ⑤ base vs 나머지 비교 → 모든 pair 비교 + Union-Find 컴포넌트

---

---
## 0. 데이터 로드

In [1]:
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

INPUT_PATH = "reviews_step1_cleaned.csv"
TEXT_COL = "리뷰내용_clean"

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"로드 완료: {len(df):,}건")
print(f"컬럼 수: {df.shape[1]}개")

로드 완료: 685,042건
컬럼 수: 49개


In [2]:
# 작성일 datetime 변환
df['작성일'] = pd.to_datetime(df['작성일'], errors='coerce')

# 작성일 결측 확인
n_nat = df['작성일'].isna().sum()
print(f"작성일 결측: {n_nat:,}건")

# 텍스트 결측 → 빈 문자열로
df[TEXT_COL] = df[TEXT_COL].fillna('').astype(str)

작성일 결측: 0건


---
## STEP 0. 탈퇴/닉변 작성자 처리

작성자가 `'-'`인 경우, 무신사 시스템상 익명 처리된 케이스이므로 각 행을 **서로 다른 사람**으로 취급한다.
→ 이후 중복 검토에서 자동으로 제외됨.

In [3]:
# '-' 작성자에 고유 ID 부여
df['작성자_norm'] = df['작성자'].astype(str)
mask_anon = df['작성자_norm'].str.strip() == "'-"

# 익명 행은 인덱스 기반 고유 ID로 변환
df.loc[mask_anon, '작성자_norm'] = (
    '_anon_' + df.loc[mask_anon].index.astype(str)
)

print(f"탈퇴/닉변 작성자: {mask_anon.sum():,}건 → 고유 ID로 변환")
print(f"고유 작성자 수: {df['작성자_norm'].nunique():,}명")

탈퇴/닉변 작성자: 3,975건 → 고유 ID로 변환
고유 작성자 수: 349,096명


---
## STEP 1. month 타입 분리

리뷰타입이 `month`인 리뷰는 한 달 사용 후 작성하는 후기로, 다른 타입(일반/스타일)과 시점·결이 다르다.
→ 중복 검토 대상에서 제외하고 그대로 보존.

In [4]:
print("리뷰타입 분포:")
print(df['리뷰타입'].value_counts(dropna=False))

mask_month = df['리뷰타입'] == 'month'
df_month = df[mask_month].copy()
df_main = df[~mask_month].copy()

print(f"\nmonth 리뷰: {len(df_month):,}건 (중복 검토 제외)")
print(f"main 리뷰: {len(df_main):,}건 (중복 검토 대상)")

리뷰타입 분포:
리뷰타입
goods         376942
photo         109292
style         104397
general        68881
month          25088
experience       442
Name: count, dtype: int64

month 리뷰: 25,088건 (중복 검토 제외)
main 리뷰: 659,954건 (중복 검토 대상)


---
## 보조 컬럼 + 코사인 유사도 함수

- **옵션키**: `(구매사이즈, 구매상세)` 튜플. NaN은 `None`으로 통일.
- **보존 기준**: `전체_글자수`(한글+영문+숫자 전체 길이) 기준. ④ 반영.
- **코사인 유사도**: TF-IDF 문자 bi-gram 벡터 기반. 15자 미만은 완전일치만 인정.
- **세션**: 세션 첫 번째 리뷰 기준 24h 윈도우 (체이닝 방지). ② 반영.
- **중복 판정**: 모든 pair 비교 후 Union-Find로 중복 컴포넌트 구성. ⑤ 반영.

In [5]:
# 옵션키 생성: (사이즈, 상세) 튜플, NaN은 None
def make_option_key(s, d):
    s_v = s if pd.notna(s) else None
    d_v = d if pd.notna(d) else None
    return (s_v, d_v)

df_main['옵션키'] = [
    make_option_key(s, d)
    for s, d in zip(df_main['구매사이즈'], df_main['구매상세'])
]

# ④ 기준 리뷰 선정: 전체_글자수 사용 (한글+영문+숫자 포함)
# 한글_글자수만 쓰면 165/52, XL, 기장 103 같은 수치 정보가 누락될 수 있음
if '전체_글자수' not in df_main.columns:
    df_main['전체_글자수'] = df_main['리뷰내용_clean'].str.len()

print(f"옵션키 고유값: {df_main['옵션키'].nunique():,}개")
print(f"전체_글자수 통계:\n{df_main['전체_글자수'].describe()}")

옵션키 고유값: 1,570개
전체_글자수 통계:
count    659954.000000
mean         44.859157
std          30.602522
min           0.000000
25%          29.000000
50%          36.000000
75%          49.000000
max        1424.000000
Name: 전체_글자수, dtype: float64


In [6]:
# 코사인 유사도 (TF-IDF 문자 bi-gram 기반)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

COSINE_THRESHOLD = 0.85
MIN_LEN_FOR_COSINE = 15  # 짧은 텍스트는 완전일치만 인정

def normalize_for_compare(s: str) -> str:
    """코사인 유사도 비교 전용 정규화: 공백·문장부호 제거.
    문장부호/마침표 한두 개 차이로 유사도가 깎이는 문제를 방지한다."""
    return re.sub(r'[\s\.\,\!\?\~\^\;\:\-\_\(\)\[\]\"\']+', '', s)

def char_bigram_analyzer(s: str):
    """TfidfVectorizer용 문자 bi-gram 분석기."""
    return [s[i:i+2] for i in range(len(s) - 1)] if len(s) >= 2 else [s]

def cosine_sim(s1: str, s2: str) -> float:
    """TF-IDF 문자 bi-gram 코사인 유사도. 정규화(공백/문장부호 제거) 후 계산.
    짧은 텍스트(15자 미만)는 완전일치만 1.0으로 인정."""
    if not s1 or not s2:
        return 0.0
    n1 = normalize_for_compare(s1)
    n2 = normalize_for_compare(s2)
    if len(n1) < MIN_LEN_FOR_COSINE or len(n2) < MIN_LEN_FOR_COSINE:
        return 1.0 if n1 == n2 else 0.0
    try:
        vec = TfidfVectorizer(analyzer=char_bigram_analyzer)
        tfidf = vec.fit_transform([n1, n2])
        return float(cosine_similarity(tfidf[0], tfidf[1])[0][0])
    except Exception:
        return 0.0

# 동작 확인
print("[코사인 유사도 함수 테스트]")
print(f"문장부호만 다름:    {cosine_sim('정말 좋아요. 잘 입을게요!', '정말 좋아요 잘 입을게요'):.3f}")
print(f"공백만 다름:        {cosine_sim('가성비 좋고 핏도 정사이즈입니다 추천드려요', '가성비좋고핏도정사이즈입니다추천드려요'):.3f}")
print(f"부분 유사:          {cosine_sim('두께감 적당하고 핏 좋습니다 추천드려요', '두께감 적당하고 색감 예쁘네요 추천'):.3f}")
print(f"완전 다름:          {cosine_sim('정말 좋아요 잘 입을게요', '사이즈가 너무 작아서 환불했습니다'):.3f}")
print(f"짧은 텍스트 동일:   {cosine_sim('좋아요', '좋아요'):.3f}")
print(f"짧은 텍스트 다름:   {cosine_sim('좋아요', '괜찮아요'):.3f}")

[코사인 유사도 함수 테스트]
문장부호만 다름:    1.000
공백만 다름:        1.000
부분 유사:          0.308
완전 다름:          0.000
짧은 텍스트 동일:   1.000
짧은 텍스트 다름:   0.000


---
## 24h 세션 분리 (체이닝 방지 적용)

같은 (작성자, 상품) 안에서 작성일 순 정렬 후, **세션 첫 번째 리뷰 기준으로 24h를 초과**하면 새 세션으로 분리.

기존 방식(인접 간격만 체크)은 00시→23시→46시처럼 체이닝이 발생할 수 있어 수정함. ② 반영.

In [7]:
# 정렬: 작성자 → 상품 → 작성일
df_main = df_main.sort_values(
    ['작성자_norm', 'goodsNo', '작성일']
).reset_index(drop=True)

# ② 체이닝 방지: 세션 첫 번째 리뷰 기준으로 24h 초과 시 새 세션 시작
# 기존: 인접 리뷰 간 시간차만 체크 → 00시->23시->46시가 같은 세션으로 묶히는 문제
# 변경: 세션 시작 시점 기준으로도 24h를 넘지 않는지 함께 확인

def assign_sessions(group: pd.DataFrame) -> pd.Series:
    times = group['작성일'].values
    sessions = []
    session_id = 0
    session_start = times[0]
    for t in times:
        diff_from_start = (t - session_start) / np.timedelta64(1, 'h')
        if diff_from_start > 24:
            session_id += 1
            session_start = t
        sessions.append(session_id)
    return pd.Series(sessions, index=group.index)

df_main['세션'] = (
    df_main.groupby(['작성자_norm', 'goodsNo'], group_keys=False)
           .apply(assign_sessions)
)

g = df_main.groupby(['작성자_norm', 'goodsNo'])
print(f"세션 부여 완료")
print(f"(작성자, 상품) 그룹 수: {g.ngroups:,}")
print(f"(작성자, 상품, 세션) 그룹 수: {df_main.groupby(['작성자_norm', 'goodsNo', '세션']).ngroups:,}")

세션 부여 완료
(작성자, 상품) 그룹 수: 469,991
(작성자, 상품, 세션) 그룹 수: 526,957


In [8]:
#pd.set_option('display.max_columns',None)
pd.options.display.max_columns = 30

In [9]:
# ③ 컬럼명 변경: is_repurchase → long_gap_review
# 주문 ID 없이 실제 재구매를 확정할 수 없으므로 더 정확한 이름으로 변경
session_per_pair = df_main.groupby(['작성자_norm', 'goodsNo'])['세션'].transform('nunique')
df_main['long_gap_review'] = session_per_pair > 1

n_long_gap_rows = df_main['long_gap_review'].sum()
n_long_gap_pairs = (session_per_pair > 1).groupby(
    [df_main['작성자_norm'], df_main['goodsNo']]
).first().sum()
print(f"long_gap_review 행 수: {n_long_gap_rows:,}건")
print(f"long_gap_review (작성자, 상품) 쌍: {n_long_gap_pairs:,}쌍")
print('※ 실제 재구매 여부는 주문 ID 없이 확정 불가. 가능성 플래그로만 활용.')

long_gap_review 행 수: 129,876건
long_gap_review (작성자, 상품) 쌍: 50,302쌍
※ 실제 재구매 여부는 주문 ID 없이 확정 불가. 가능성 플래그로만 활용.


---
## 그룹 단위 중복 처리 (Step 2 ~ 4)

`(작성자_norm, goodsNo, 세션)` 단위로 묶고:

1. 그룹 크기 = 1 → 단독 리뷰 (그대로 keep)
2. 그룹 크기 >= 2:
   - **모든 pair를 비교** (① + ⑤ 반영)
   - Union-Find로 중복 pair를 컴포넌트 단위로 묶음
   - 컴포넌트 내 `전체_글자수` 가장 많은 리뷰를 base로 선정, 나머지 drop
   - 중복 아닌 리뷰는 각각 keep

In [10]:
def union_find_components(n: int, dup_pairs: list) -> list:
    """
    Union-Find로 중복 pair들을 컴포넌트(클러스터)로 묶는다.

    Parameters
    ----------
    n         : 그룹 내 리뷰 수
    dup_pairs : [(i, j), ...] 중복으로 판정된 인덱스 쌍 목록

    Returns
    -------
    List[int] : 각 리뷰의 컴포넌트 ID (같은 ID = 같은 중복 클러스터)
    """
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        parent[find(x)] = find(y)

    for i, j in dup_pairs:
        union(i, j)

    return [find(i) for i in range(n)]


def process_group(group: pd.DataFrame) -> pd.DataFrame:
    """
    같은 (작성자, 상품, 세션) 그룹 내 다중 리뷰 처리.

    변경 사항:
      - same_option_same_type도 코사인 유사도 체크 후 판단 (① 반영)
      - base 선정 기준: 전체_글자수 (④ 반영)
      - 모든 pair 비교 + Union-Find 컴포넌트 방식 (⑤ 반영)

    반환 컬럼:
      - action   : 'keep' | 'drop'
      - dup_flag : 플래그 문자열
      - is_base  : 컴포넌트 내 대표 리뷰 여부
      - kept_id  : 해당 컴포넌트의 대표 리뷰번호
    """
    g = group.copy().reset_index(drop=False)  # 원본 인덱스 보존
    n = len(g)

    if n == 1:
        g['action']   = 'keep'
        g['dup_flag'] = 'unique'
        g['is_base']  = True
        g['kept_id']  = g['리뷰번호'].iloc[0]
        return g.set_index('index')

    texts   = g['리뷰내용_clean'].tolist()
    options = g['옵션키'].tolist()
    rtypes  = g['리뷰타입'].tolist()

    # ── 모든 pair 비교 ────────────────────────────────────────────────
    dup_pairs  = []   # (i, j) 중복 pair
    pair_flags = {}   # (i, j) -> flag 문자열

    for i in range(n):
        for j in range(i + 1, n):
            same_opt = (options[i] == options[j])
            same_typ = (rtypes[i]  == rtypes[j])
            sim      = cosine_sim(texts[i], texts[j])

            if same_opt and same_typ:
                # ① 수정: 텍스트 유사도 확인 후 판단
                if sim >= COSINE_THRESHOLD:
                    flag = 'same_option_same_type_dup'
                    dup_pairs.append((i, j))
                else:
                    flag = 'same_option_same_type_unique'
            elif (not same_opt) and same_typ:
                if sim >= COSINE_THRESHOLD:
                    flag = 'multi_option_dup'
                    dup_pairs.append((i, j))
                else:
                    flag = 'multi_option_unique'
            elif same_opt and (not same_typ):
                if sim >= COSINE_THRESHOLD:
                    flag = 'multi_type_dup'
                    dup_pairs.append((i, j))
                else:
                    flag = 'multi_type_unique'
            else:
                if sim >= COSINE_THRESHOLD:
                    flag = 'multi_both_dup'
                    dup_pairs.append((i, j))
                else:
                    flag = 'multi_both_unique'

            pair_flags[(i, j)] = flag

    # ── Union-Find로 중복 컴포넌트 구성 ──────────────────────────────
    comp_ids = union_find_components(n, dup_pairs)

    # ── 컴포넌트별 대표(base) 선정: 전체_글자수 최대 → 동률 시 작성일 빠른 것
    # ④ 반영: 한글_글자수 -> 전체_글자수
    g_sorted = g.sort_values(['전체_글자수', '작성일'], ascending=[False, True])

    comp_base = {}  # comp_id -> 그룹 내 위치(0~n-1)
    for pos in g_sorted.index:
        cid = comp_ids[pos]
        if cid not in comp_base:
            comp_base[cid] = pos

    # ── 각 행에 action / dup_flag / is_base / kept_id 부여 ───────────
    actions  = []
    flags    = []
    is_bases = []
    kept_ids = []

    for pos in range(n):
        cid      = comp_ids[pos]
        base_pos = comp_base[cid]
        is_base  = (pos == base_pos)
        kept_id  = g.loc[base_pos, '리뷰번호']

        if is_base:
            action = 'keep'
            # base의 dup_flag: 이 컴포넌트에서 발생한 _dup 플래그 중 첫 번째
            comp_dup_flags = [
                pair_flags[(min(i, j), max(i, j))]
                for (i, j) in dup_pairs
                if comp_ids[i] == cid
            ]
            if comp_dup_flags:
                flag = next((f for f in comp_dup_flags if f.endswith('_dup')),
                            comp_dup_flags[0])
            else:
                flag = 'unique'
        else:
            # 동반자: base와의 pair flag 사용
            i, j = (min(pos, base_pos), max(pos, base_pos))
            flag = pair_flags.get((i, j))
            if flag is None:
                # Union-Find로 간접 연결된 케이스: 컴포넌트의 첫 번째 _dup flag 사용
                comp_dup_flags = [
                    pair_flags[(min(a, b), max(a, b))]
                    for (a, b) in dup_pairs
                    if comp_ids[a] == cid
                ]
                flag = comp_dup_flags[0] if comp_dup_flags else 'same_option_same_type_dup'
            action = 'drop' if flag.endswith('_dup') else 'keep'

        actions.append(action)
        flags.append(flag)
        is_bases.append(is_base)
        kept_ids.append(kept_id)

    g['action']   = actions
    g['dup_flag'] = flags
    g['is_base']  = is_bases
    g['kept_id']  = kept_ids

    return g.set_index('index')

In [11]:
# 그룹 사이즈 사전 확인
group_sizes = df_main.groupby(['작성자_norm', 'goodsNo', '세션']).size()
print(f"전체 (작성자, 상품, 세션) 그룹 수: {len(group_sizes):,}")
print(f"  단일 리뷰 그룹: {(group_sizes == 1).sum():,}")
print(f"  다중 리뷰 그룹(≥2): {(group_sizes >= 2).sum():,}")
print(f"\n다중 리뷰 그룹 크기 분포:")
print(group_sizes[group_sizes >= 2].value_counts().sort_index().head(10))

전체 (작성자, 상품, 세션) 그룹 수: 526,957
  단일 리뷰 그룹: 425,413
  다중 리뷰 그룹(≥2): 101,544

다중 리뷰 그룹 크기 분포:
2    72027
3    28358
4      735
5      101
6      309
7        5
8        2
9        7
Name: count, dtype: int64


In [12]:
# 단일 그룹 / 다중 그룹 분리 처리 (성능 최적화)
df_main = df_main.merge(
    group_sizes.rename('그룹크기').reset_index(),
    on=['작성자_norm', 'goodsNo', '세션'],
    how='left'
)

mask_single = df_main['그룹크기'] == 1
df_single = df_main[mask_single].copy()
df_multi  = df_main[~mask_single].copy()

print(f"단일 그룹 행: {len(df_single):,}")
print(f"다중 그룹 행: {len(df_multi):,}")

# 단일 그룹: 일괄 처리
df_single['action']   = 'keep'
df_single['dup_flag'] = 'unique'
df_single['is_base']  = True
df_single['kept_id']  = df_single['리뷰번호']

# 다중 그룹: process_group 적용
tqdm.pandas(desc='다중 그룹 처리')
df_multi_processed = (
    df_multi.groupby(['작성자_norm', 'goodsNo', '세션'], group_keys=False, sort=False)
            .progress_apply(process_group)
)

# pandas 버전 호환: groupby 키 컬럼이 결과에서 빠진 경우(pandas 3.0+) 복구
for col in ['작성자_norm', 'goodsNo', '세션']:
    if col not in df_multi_processed.columns:
        df_multi_processed[col] = df_multi.loc[df_multi_processed.index, col]

# 합치기
df_main_processed = pd.concat(
    [df_single, df_multi_processed], ignore_index=True
).sort_values(['작성자_norm', 'goodsNo', '작성일']).reset_index(drop=True)

print(f"\n처리 완료: {len(df_main_processed):,}건")

단일 그룹 행: 425,413
다중 그룹 행: 234,541


다중 그룹 처리:   0%|          | 0/101544 [00:00<?, ?it/s]


처리 완료: 659,954건


---
## 처리 결과 통계

In [13]:
print('[action 분포]')
print(df_main_processed['action'].value_counts())
print(f'\n[dup_flag 분포]')
print(df_main_processed['dup_flag'].value_counts(dropna=False))
print(f'\n[long_gap_review 분포]')
print(df_main_processed['long_gap_review'].value_counts())

[action 분포]
action
keep    600717
drop     59237
Name: count, dtype: int64

[dup_flag 분포]
dup_flag
unique                       556378
multi_type_dup                98569
multi_option_dup               3785
multi_both_dup                  681
same_option_same_type_dup       498
multi_type_unique                41
multi_option_unique               1
multi_both_unique                 1
Name: count, dtype: int64

[long_gap_review 분포]
long_gap_review
False    530078
True     129876
Name: count, dtype: int64


In [14]:
# 샘플 확인: drop된 리뷰 vs 보존된 기준 리뷰 페어
print("[multi_option_dup 샘플]")
sample_dup = df_main_processed[
    (df_main_processed['dup_flag'] == 'multi_option_dup') &
    (df_main_processed['action'] == 'drop')
].head(5)
for _, row in sample_dup.iterrows():
    base = df_main_processed[df_main_processed['리뷰번호'] == row['kept_id']].iloc[0]
    print(f"\n  [기준] {row['kept_id']} | 옵션={base['옵션키']} | 한글_글자수={base['한글_글자수']}")
    print(f"     → {base[TEXT_COL][:80]}")
    print(f"  [drop] {row['리뷰번호']} | 옵션={row['옵션키']} | 한글_글자수={row['한글_글자수']}")
    print(f"     → {row[TEXT_COL][:80]}")

[multi_option_dup 샘플]

  [기준] 15343006 | 옵션=('M', '셔츠딥그레이') | 한글_글자수=71
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합
  [drop] 15342958 | 옵션=('M', '셔츠라이트베이지') | 한글_글자수=71
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합

  [기준] 15343006 | 옵션=('M', '셔츠딥그레이') | 한글_글자수=71
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합
  [drop] 15342623 | 옵션=('M', '셔츠블랙') | 한글_글자수=71
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합

  [기준] 42693190 | 옵션=('2XL', '베이지') | 한글_글자수=43
     → 옷감이 가벼워 여름에 시원하게 입을수 있겠네요뒷주머니가 사진에 안보였는데 진짜로 없어 좀 아쉽네요;;
  [drop] 42693216 | 옵션=('M', '베이지') | 한글_글자수=43
     → 옷감이 가벼워 여름에 시원하게 입을수 있겠네요뒷주머니가 사진에 안보였는데 진짜로 없어 좀 아쉽네요;;

  [기준] 44590746 | 옵션=('XL', '차콜') | 한글_글자수=27
     → 다른색들도 다 맘에들어서 더 샀는데 역시 맘에드네요 감사합니다ㅎㅎ
  [drop] 44590756 | 옵션=('XL', '블랙') | 한글_글자수=27
     → 다른색들도 다 맘에들어서 더 샀는데 역시 맘에드네요 감사합니다ㅎㅎ

  [기준] 44590746

In [15]:
print("[multi_option_unique 샘플 - 둘 다 보존되는 케이스]")
# 동반자(is_base=False)만 순회 → 페어가 한 번씩만 출력됨
companions = (df_main_processed[
    (df_main_processed['dup_flag'] == 'multi_option_unique') & (~df_main_processed['is_base'])
].head(5))
for _, row in companions.iterrows():
    base = df_main_processed[df_main_processed['리뷰번호'] == row['kept_id']].iloc[0]
    print(f"\n  [기준]      {row['kept_id']} | 옵션={base['옵션키']} | 한글_글자수={base['한글_글자수']}")
    print(f"     → {base[TEXT_COL][:100]}")
    print(f"  [같이 보존] {row['리뷰번호']} | 옵션={row['옵션키']} | 한글_글자수={row['한글_글자수']}")
    print(f"     → {row[TEXT_COL][:100]}")

[multi_option_unique 샘플 - 둘 다 보존되는 케이스]

  [기준]      3954491 | 옵션=('LARGE', '[NO기모]') | 한글_글자수=54
     → 너무이쁩니다 2번사세요 커플룩으로 삿는데 저희 둘다 반해가지구 다른색도 살 예정입니다 ㅋㅋ색감깡패 오집니다ㅋㅋ초록색도 기대되네요
  [같이 보존] 3954484 | 옵션=('MEDIUM', '[NO기모]') | 한글_글자수=45
     → 너무이쁩니다 2번사세요 커플룩으로 삿는데ㅋㅋ 저희 둘다 반해가지구 다른색도 살 예정입니다 ㅋㅋ색감깡패 오집니다


---
## month 리뷰 통합 + 저장

- `reviews_step2_dedup.csv`: 최종 보존 리뷰 (Step 3 임베딩 입력)
- `reviews_step2_dropped.csv`: drop된 리뷰 (검증용 보관)

In [16]:
# month 리뷰는 그대로 keep, 플래그만 추가
df_month['action']          = 'keep'
df_month['dup_flag']        = 'month_excluded'
df_month['is_base']         = True
df_month['kept_id']         = df_month['리뷰번호']
df_month['long_gap_review'] = False
df_month['세션']             = 0
df_month['그룹크기']         = 1
df_month['옵션키'] = [
    make_option_key(s, d)
    for s, d in zip(df_month['구매사이즈'], df_month['구매상세'])
]
if '전체_글자수' not in df_month.columns:
    df_month['전체_글자수'] = df_month['리뷰내용_clean'].str.len()

# 컬럼 정렬을 df_main_processed에 맞춤
df_month = df_month[df_main_processed.columns]

# 통합
df_final = pd.concat([df_main_processed, df_month], ignore_index=True)
print(f'통합 후: {len(df_final):,}건')

통합 후: 685,042건


In [17]:
# 최종 저장: keep만
df_keep = df_final[df_final['action'] == 'keep'].copy()
df_drop = df_final[df_final['action'] == 'drop'].copy()

print(f"최종 보존: {len(df_keep):,}건 ({len(df_keep)/len(df_final)*100:.2f}%)")
print(f"제거된 중복: {len(df_drop):,}건 ({len(df_drop)/len(df_final)*100:.2f}%)")

print(f"\n[보존 리뷰 dup_flag 분포]")
print(df_keep['dup_flag'].value_counts())

print(f"\n[제거된 리뷰 dup_flag 분포]")
print(df_drop['dup_flag'].value_counts())

최종 보존: 625,805건 (91.35%)
제거된 중복: 59,237건 (8.65%)

[보존 리뷰 dup_flag 분포]
dup_flag
unique                       556378
multi_type_dup                42299
month_excluded                25088
multi_option_dup               1711
same_option_same_type_dup       223
multi_both_dup                   63
multi_type_unique                41
multi_option_unique               1
multi_both_unique                 1
Name: count, dtype: int64

[제거된 리뷰 dup_flag 분포]
dup_flag
multi_type_dup               56270
multi_option_dup              2074
multi_both_dup                 618
same_option_same_type_dup      275
Name: count, dtype: int64


#### 플래그 전체 목록

**판정결과**: `_dup` (유사도 >= 0.85, 중복) / `_unique` (유사도 < 0.85, 다른 내용)

| 플래그 | 조건 | action |
|--------|------|--------|
| `unique` | 그룹 내 단독 리뷰 | keep |
| `month_excluded` | 한 달 사용 후기 | keep |
| `same_option_same_type_dup` | 같은 옵션+타입 + 유사도 >= 0.85 | drop |
| `same_option_same_type_unique` | 같은 옵션+타입 + 유사도 < 0.85 | keep |
| `multi_option_dup` | 다른 옵션+같은 타입 + 유사도 >= 0.85 | drop |
| `multi_option_unique` | 다른 옵션+같은 타입 + 유사도 < 0.85 | keep |
| `multi_type_dup` | 같은 옵션+다른 타입 + 유사도 >= 0.85 | drop |
| `multi_type_unique` | 같은 옵션+다른 타입 + 유사도 < 0.85 | keep |
| `multi_both_dup` | 다른 옵션+다른 타입 + 유사도 >= 0.85 | drop |
| `multi_both_unique` | 다른 옵션+다른 타입 + 유사도 < 0.85 | keep |

**변경 사항 요약**
- ① `same_option_same_type`에 유사도 체크 추가 + `_unique` 플래그 신규
- ② 세션 분리: 인접 간격 기준 → 세션 시작 기준 24h 윈도우 (체이닝 방지)
- ③ `is_repurchase` → `long_gap_review`
- ④ base 선정: `한글_글자수` → `전체_글자수`
- ⑤ base vs 나머지 → 모든 pair 비교 + Union-Find 컴포넌트

In [18]:
# CSV 저장
OUTPUT_KEEP = "reviews_step2_dedup.csv"
OUTPUT_DROP = "reviews_step2_dropped.csv"

df_keep.to_csv(OUTPUT_KEEP, index=False)
df_drop.to_csv(OUTPUT_DROP, index=False)

print(f"저장 완료:")
print(f"  - {OUTPUT_KEEP} ({len(df_keep):,}건)")
print(f"  - {OUTPUT_DROP} ({len(df_drop):,}건)")

저장 완료:
  - reviews_step2_dedup.csv (625,805건)
  - reviews_step2_dropped.csv (59,237건)


---
## (참고) 검증용 분석

전체 처리 결과를 한눈에 점검하기 위한 통계.
실제 분석에는 사용하지 않으므로 필요 시 셀 단위로 참고.

In [19]:
# 옵션 다중 구매 케이스 통계
multi_opt_pairs = df_keep[df_keep['dup_flag'].isin([
    'multi_option_unique', 'multi_option_dup'
])]
print(f'옵션 다중 구매 관련 보존된 리뷰: {len(multi_opt_pairs):,}건')
print(f'  - base: {multi_opt_pairs["is_base"].sum():,}건')
print(f'  - 동반자(unique 케이스): {(~multi_opt_pairs["is_base"]).sum():,}건')

# long_gap_review 통계
long_gap_keep = df_keep[df_keep['long_gap_review']]
print(f'\nlong_gap_review 케이스 보존 리뷰: {len(long_gap_keep):,}건')
print(f'long_gap_review (작성자, 상품) 쌍: {long_gap_keep.groupby(["작성자_norm", "goodsNo"]).ngroups:,}쌍')
print('※ 실제 재구매 여부는 주문 ID 없이 확정 불가.')

if '브랜드' in df_keep.columns and '카테고리' in df_keep.columns:
    print(f'\n[브랜드 x 카테고리별 보존 리뷰 수]')
    print(df_keep.groupby(['브랜드', '카테고리']).size().unstack(fill_value=0))

옵션 다중 구매 관련 보존된 리뷰: 1,712건
  - base: 1,711건
  - 동반자(unique 케이스): 1건

long_gap_review 케이스 보존 리뷰: 120,997건
long_gap_review (작성자, 상품) 쌍: 50,302쌍
※ 실제 재구매 여부는 주문 ID 없이 확정 불가.


In [20]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

SAMPLE_N     = 3
TEXT_PREVIEW = 120

def print_pair_samples(flag_name, source_df, n=SAMPLE_N):
    print(f"\n{'='*95}")
    print(f'[dup_flag = {flag_name}]')
    print('='*95)
    companions = source_df[
        (source_df['dup_flag'] == flag_name) & (~source_df['is_base'].fillna(False))
    ]
    if len(companions) == 0:
        print('  (해당 플래그의 동반자 행이 없음)')
        return
    sampled = companions.sample(n=min(n, len(companions)), random_state=42)
    for k, (_, comp) in enumerate(sampled.iterrows(), 1):
        base_match = source_df[source_df['리뷰번호'] == comp['kept_id']]
        if len(base_match) == 0:
            continue
        base = base_match.iloc[0]
        same_opt = base['옵션키'] == comp['옵션키']
        same_typ = base['리뷰타입'] == comp['리뷰타입']
        print(f"\n  - 페어 {k} - (옵션 {'동일' if same_opt else '다름'} / 타입 {'동일' if same_typ else '다름'})")
        print(f"    [base]   리뷰번호={base['리뷰번호']} | dup_flag={base['dup_flag']} | action={base['action']}")
        print(f"             옵션={base['옵션키']} | 타입={base['리뷰타입']} | 전체_글자수={base['전체_글자수']}")
        print(f"             -> {str(base['리뷰내용_clean'])[:TEXT_PREVIEW]}")
        print(f"    [동반자] 리뷰번호={comp['리뷰번호']} | dup_flag={comp['dup_flag']} | action={comp['action']}")
        print(f"             옵션={comp['옵션키']} | 타입={comp['리뷰타입']} | 전체_글자수={comp['전체_글자수']}")
        print(f"             -> {str(comp['리뷰내용_clean'])[:TEXT_PREVIEW]}")

def print_solo_samples(flag_name, source_df, n=SAMPLE_N):
    print(f"\n{'='*95}")
    print(f'[dup_flag = {flag_name}]')
    print('='*95)
    pool = source_df[source_df['dup_flag'] == flag_name]
    if len(pool) == 0:
        print('  (해당 플래그 없음)')
        return
    sampled = pool.sample(n=min(n, len(pool)), random_state=42)
    for k, (_, row) in enumerate(sampled.iterrows(), 1):
        print(f'\n  - 샘플 {k} -')
        print(f"    리뷰번호={row['리뷰번호']} | dup_flag={row['dup_flag']} | action={row['action']}")
        print(f"    옵션={row['옵션키']} | 타입={row['리뷰타입']} | 전체_글자수={row['전체_글자수']}")
        print(f"    -> {str(row['리뷰내용_clean'])[:TEXT_PREVIEW]}")

source = df_final

# _dup 플래그
for flag in ['same_option_same_type_dup', 'multi_type_dup', 'multi_option_dup', 'multi_both_dup']:
    print_pair_samples(flag, source)

# _unique 플래그 (신규: same_option_same_type_unique 포함)
for flag in ['same_option_same_type_unique', 'multi_type_unique', 'multi_option_unique', 'multi_both_unique']:
    print_pair_samples(flag, source)

# 단독 플래그
for flag in ['unique', 'month_excluded']:
    print_solo_samples(flag, source)


[dup_flag = same_option_same_type_dup]

  - 페어 1 - (옵션 동일 / 타입 동일)
    [base]   리뷰번호=66051602 | dup_flag=same_option_same_type_dup | action=keep
             옵션=('M', '(린넨)') | 타입=general | 전체_글자수=29
             -> 무난하게 예쁘고깔끔합니다. 흰셔츠는 이것만 사입어요.
    [동반자] 리뷰번호=66051640 | dup_flag=same_option_same_type_dup | action=drop
             옵션=('M', '(린넨)') | 타입=general | 전체_글자수=29
             -> 무난하게 예쁘고깔끔합니다. 흰셔츠는 이것만 사입어요.

  - 페어 2 - (옵션 동일 / 타입 동일)
    [base]   리뷰번호=57088330 | dup_flag=same_option_same_type_dup | action=keep
             옵션=('XL', '(논기모)') | 타입=goods | 전체_글자수=27
             -> 데일리로 입기에 너무 좋아요 빠른 배송 감사합니다
    [동반자] 리뷰번호=57088347 | dup_flag=same_option_same_type_dup | action=drop
             옵션=('XL', '(논기모)') | 타입=goods | 전체_글자수=27
             -> 데일리로 입기에 너무 좋아요 빠른 배송 감사합니다

  - 페어 3 - (옵션 동일 / 타입 동일)
    [base]   리뷰번호=18218726 | dup_flag=same_option_same_type_dup | action=keep
             옵션=('LARGE', None) | 타입=goods | 전체_글자수=26
             -> 너무 좋아요 역시 티는 검정이죠 추천드

In [21]:
# 그룹 크기별 분포 확인
print("[(작성자, 상품, 세션) 그룹 크기 분포]")
print(df_final[df_final['리뷰타입'] != 'month']
      .groupby(['작성자_norm', 'goodsNo', '세션']).size()
      .value_counts().sort_index())

[(작성자, 상품, 세션) 그룹 크기 분포]
1    425413
2     72027
3     28358
4       735
5       101
6       309
7         5
8         2
9         7
Name: count, dtype: int64


In [22]:
# 그룹 크기 ≥ 3인 케이스만 추출해서 그룹 단위로 모든 멤버 표시

LARGE_GROUP_SAMPLE_N = 5  # 출력할 그룹 수

def print_large_group_samples(source_df, min_size=3, n=LARGE_GROUP_SAMPLE_N):
    """그룹 사이즈 ≥ min_size 인 케이스를 그룹 단위로 출력. 모든 멤버를 한 번에 보여줌."""
    
    # month는 별도 처리되므로 제외
    df_check = source_df[source_df['리뷰타입'] != 'month'].copy()
    
    # 그룹 크기 계산
    sizes = df_check.groupby(['작성자_norm', 'goodsNo', '세션']).size()
    large_groups = sizes[sizes >= min_size]
    
    if len(large_groups) == 0:
        print(f"\n그룹 크기 ≥ {min_size}인 케이스 없음")
        return
    
    print(f"\n{'='*95}")
    print(f"[그룹 크기 ≥ {min_size} 케이스: 총 {len(large_groups):,}그룹]")
    print('='*95)
    
    # 그룹 크기 분포
    print("\n그룹 크기별 개수:")
    print(large_groups.value_counts().sort_index().to_string())
    
    # 무작위 n개 그룹 선택
    sample_keys = large_groups.sample(n=min(n, len(large_groups)), 
                                       random_state=42).index.tolist()
    
    for gi, key in enumerate(sample_keys, 1):
        author, goods, sess = key
        members = df_check[
            (df_check['작성자_norm'] == author) &
            (df_check['goodsNo'] == goods) &
            (df_check['세션'] == sess)
        ].sort_values(['is_base', '한글_글자수'], ascending=[False, False])
        
        # 이 그룹 안의 dup_flag 종류 한눈에
        flag_summary = members['dup_flag'].value_counts().to_dict()
        
        print(f"\n{'─'*95}")
        print(f"그룹 {gi}: 작성자={author} | 상품={goods} | 세션={sess} | 멤버 {len(members)}명")
        print(f"  플래그 구성: {flag_summary}")
        print(f"  base 리뷰번호: {members[members['is_base']]['리뷰번호'].iloc[0]}")
        print(f"{'─'*95}")
        
        for mi, (_, m) in enumerate(members.iterrows(), 1):
            role = "★ base" if m['is_base'] else "  동반자"
            print(f"\n  [{role}] 멤버 {mi}")
            print(f"    리뷰번호={m['리뷰번호']} | dup_flag={m['dup_flag']} | "
                  f"is_base={m['is_base']} | action={m['action']}")
            print(f"    옵션={m['옵션키']} | 타입={m['리뷰타입']} | "
                  f"한글_글자수={m['한글_글자수']} | 작성일={m['작성일']}")
            print(f"    → {str(m[TEXT_COL])[:TEXT_PREVIEW]}")


# 그룹 크기별로 따로 보기
print_large_group_samples(source, min_size=3, n=5)   # 3명 이상 그룹 5개
print_large_group_samples(source, min_size=4, n=3)   # 4명 이상 그룹 3개
print_large_group_samples(source, min_size=5, n=2)   # 5명 이상 그룹 2개 (있다면)


[그룹 크기 ≥ 3 케이스: 총 29,517그룹]

그룹 크기별 개수:
3    28358
4      735
5      101
6      309
7        5
8        2
9        7

───────────────────────────────────────────────────────────────────────────────────────────────
그룹 1: 작성자=배택환 | 상품=2085369 | 세션=0 | 멤버 3명
  플래그 구성: {'unique': 3}
  base 리뷰번호: 32055090
───────────────────────────────────────────────────────────────────────────────────────────────

  [★ base] 멤버 1
    리뷰번호=32055090 | dup_flag=unique | is_base=True | action=keep
    옵션=('L', '네이비') | 타입=style | 한글_글자수=36 | 작성일=2022-09-05 18:11:03
    → 전체적으로 폼이 큽니다 팔부분에 시보리가 강해서 팔 기장이 길어도 이쁘게 떨어지네요

  [★ base] 멤버 2
    리뷰번호=32054850 | dup_flag=unique | is_base=True | action=keep
    옵션=('L', '네이비') | 타입=photo | 한글_글자수=24 | 작성일=2022-09-05 18:04:25
    → 상당히 오버한 맨투맨 입니다 가성비 좋게 잘 입을 것 같네요

  [★ base] 멤버 3
    리뷰번호=32054879 | dup_flag=unique | is_base=True | action=keep
    옵션=('L', '네이비') | 타입=goods | 한글_글자수=23 | 작성일=2022-09-05 18:05:33
    → 가성비 좋은 맨투맨 입니다 가을에 잘 입고 다닐 것 같아요

───────────────

---
## ⚠️ 알려진 한계: base 행의 dup_flag 의미 모호함

현재 구조에서 그룹 내 여러 동반자가 있고 각 동반자가 서로 다른 dup_flag를 가지는 경우, base 행의 dup_flag는 **그룹 내 첫 번째로 발견된 `_dup` 플래그를 그대로 사용**한다.

### 예시
한 그룹에 멤버 6명이 있고:
- base ↔ 동반자 2: `multi_both_dup` (다른 옵션 + 다른 타입)
- base ↔ 동반자 4: `multi_option_dup` (다른 옵션 + 같은 타입)
- base ↔ 동반자 5: `multi_type_dup` (같은 옵션 + 다른 타입)

이때 base 행의 `dup_flag`는 첫 번째인 `multi_both_dup`으로 표시되지만, 실제로 base는 다른 종류의 관계도 함께 가지고 있다.

### 영향
- **알고리즘 동작에는 문제 없음**: base/동반자 구분은 `is_base` 컬럼이 정확히 처리한다.
- **통계 시 주의**: `df['dup_flag'] == 'multi_both_dup'`로 단순 필터링하면 base와 동반자가 섞이며, base 카운트가 misleading할 수 있다.

### 권장 분석 패턴
페어/플래그 단위 분석 시 **항상 동반자 기준으로 필터링**한다:

```python
# 정확한 multi_both_dup 페어 카운트
df[(df['dup_flag'] == 'multi_both_dup') & (~df['is_base'])]
```

base의 정보가 필요하면 `kept_id`로 매칭하여 가져온다.

### 왜 그대로 두는가
1. 임베딩(Step 3) 입력은 `action == 'keep'`인 모든 행이며, dup_flag 분류 자체에 의존하지 않는다.
2. 분석 단계에서 동반자 기준 필터링 패턴을 지키면 모호함이 발생하지 않는다.

In [23]:
df_keep.head()

,리뷰번호,goodsNo,리뷰타입,작성자,리뷰내용,평점,체험단,구매옵션,키,몸무게,성별,작성일,만족도,사진유무,도움돼요,...,일평균_도움돼요지수,도움여부,리뷰내용_clean,한글_글자수,전체_글자수,is_valid_for_topic,작성자_norm,옵션키,세션,long_gap_review,그룹크기,action,dup_flag,is_base,kept_id
0,34612047,1733275,goods,!!!!!!!!?,요즘 입기 좋은 것 같아요 무난무난하게 잘 입고있습니다,5.0,False,다크그레이 · L,166.0,63.0,여성,2022-11-07 18:13:53,NaN,False,0,...,0.0,0,요즘 입기 좋은 것 같아요 무난무난하게 잘 입고있습니다,23,30,True,!!!!!!!!?,"(L, 다크그레이)",0,False,1,keep,unique,True,34612047
1,51564285,3070563,goods,!!!!!!!!?,잠옷용으로 구매했어요 편하고 그냥 입기에도 조아요,5.0,False,블랙 · 2XL,180.0,90.0,missing,2023-11-23 15:50:39,NaN,False,0,...,0.0,0,잠옷용으로 구매했어요 편하고 그냥 입기에도 조아요,22,27,True,!!!!!!!!?,"(2XL, 블랙)",0,False,1,keep,unique,True,51564285
2,46679669,3251750,goods,!!!!!!!!?,잠옷용으로 휘뚜루마뚜루 입으려고 좀 크게 사긴했는데 진짜 많이 커요 조금은 두꺼운 감이 있어요,5.0,False,블랙 · XL,166.0,50.0,여성,2023-08-01 15:28:26,NaN,False,0,...,0.0,0,잠옷용으로 휘뚜루마뚜루 입으려고 좀 크게 사긴했는데 진짜 많이 커요 조금은 두꺼운 감이 있어요,40,52,True,!!!!!!!!?,"(XL, 블랙)",0,True,1,keep,unique,True,46679669
3,49013088,3251750,goods,!!!!!!!!?,잠옷으로 입으려고 크게 사긴했는데 정말 크고 두꺼워요,5.0,False,블랙 · XL,166.0,50.0,여성,2023-10-01 20:02:46,NaN,False,0,...,0.0,0,잠옷으로 입으려고 크게 사긴했는데 정말 크고 두꺼워요,23,29,True,!!!!!!!!?,"(XL, 블랙)",1,True,1,keep,unique,True,49013088
4,12731504,850153,goods,!!!!!!-,이뻐요. 자주 입겠네요.. 굿 감사랍니다 고맙습니다,5.0,False,블랙 · L,178.0,74.0,남성,2020-11-03 18:57:52,NaN,False,0,...,0.0,0,이뻐요. 자주 입겠네요.. 굿 감사랍니다 고맙습니다,20,28,True,!!!!!!-,"(L, 블랙)",0,False,1,keep,unique,True,12731504
